## First small model

### Install needed libraries

In [2]:
pip install torch torchvision pandas numpy scikit-learn


^C
Note: you may need to restart the kernel to use updated packages.


### Load data

In [9]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import laspy
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils import resample

path_flo = "C:/Users/flori/Desktop/pa-1_data/zurich_1.las"

las = laspy.read(path_flo)
df = pd.DataFrame({
    "X": las.x,
    "Y": las.y,
    "Z": las.z,
    "Intensity": las.intensity,
    "Classification": las.classification
})

# Show the first few rows
print(df.head())

                                                   X  \
0  [2683033.91, 2683074.91, 2683086.29, 2683135.9...   
1  [2683033.91, 2683074.91, 2683086.29, 2683135.9...   
2  [2683033.91, 2683074.91, 2683086.29, 2683135.9...   
3  [2683033.91, 2683074.91, 2683086.29, 2683135.9...   
4  [2683033.91, 2683074.91, 2683086.29, 2683135.9...   

                                                   Y  \
0  [1247000.0, 1247000.0, 1247000.0, 1247000.0, 1...   
1  [1247000.0, 1247000.0, 1247000.0, 1247000.0, 1...   
2  [1247000.0, 1247000.0, 1247000.0, 1247000.0, 1...   
3  [1247000.0, 1247000.0, 1247000.0, 1247000.0, 1...   
4  [1247000.0, 1247000.0, 1247000.0, 1247000.0, 1...   

                                                   Z  Intensity  \
0  [433.53000000000003, 426.62, 433.13, 417.98, 4...         71   
1  [433.53000000000003, 426.62, 433.13, 417.98, 4...         73   
2  [433.53000000000003, 426.62, 433.13, 417.98, 4...        114   
3  [433.53000000000003, 426.62, 433.13, 417.98, 4...      

### Create model

In [10]:
# Compute ground height (approximate lowest Z value)
ground_height = np.min(las.z)

# Extract Features (XYZ, Intensity, Return Number, Height Above Ground)
X = np.vstack((
    las.x, 
    las.y, 
    las.z, 
    las.intensity, 
    las.return_number,  # Additional feature
    las.z - ground_height  # Height normalization (Z - Ground)
)).T.astype(np.float32)  # Convert to float32

# Extract classification labels and create a mapping
unique_labels = np.unique(las.classification)  # Get unique class values
label_map = {label: idx for idx, label in enumerate(unique_labels)}  # Map to 0-based index
y = np.array([label_map[label] for label in las.classification]).astype(np.int64)  # Convert labels

# Take a larger random sample (e.g., 5 million points)
sample_indices = np.random.choice(X.shape[0], size=5000000, replace=False)
X = X[sample_indices]
y = y[sample_indices]

# === Balance Classes (Undersample Majority Classes) ===
min_class_count = min(np.bincount(y))  # Find smallest class size
balanced_X, balanced_y = [], []
for class_label in np.unique(y):
    class_indices = np.where(y == class_label)[0]
    sampled_indices = resample(class_indices, replace=True, n_samples=min_class_count, random_state=42)
    balanced_X.append(X[sampled_indices])
    balanced_y.append(y[sampled_indices])

X = np.vstack(balanced_X)
y = np.hstack(balanced_y)

# === Normalize Features ===
scaler = StandardScaler()
X = scaler.fit_transform(X)

# === Convert to PyTorch tensors ===
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)

# === Define Improved Neural Network ===
class PointCloudNet(nn.Module):
    def __init__(self, input_size, num_classes):
        super(PointCloudNet, self).__init__()
        self.fc1 = nn.Linear(input_size, 128)
        self.bn1 = nn.BatchNorm1d(128)
        self.fc2 = nn.Linear(128, 256)
        self.bn2 = nn.BatchNorm1d(256)
        self.fc3 = nn.Linear(256, 128)
        self.bn3 = nn.BatchNorm1d(128)
        self.fc4 = nn.Linear(128, num_classes)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(p=0.3)  # Dropout to prevent overfitting

    def forward(self, x):
        x = self.relu(self.bn1(self.fc1(x)))
        x = self.relu(self.bn2(self.fc2(x)))
        x = self.relu(self.bn3(self.fc3(x)))
        x = self.dropout(x)  # Dropout applied before final layer
        x = self.fc4(x)
        return x

# Initialize model
input_size = X_train.shape[1]  # Now using all features
num_classes = len(unique_labels)  # Number of unique classifications
model = PointCloudNet(input_size, num_classes)

# === Define Loss, Optimizer, and Learning Rate Scheduler ===
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)  # AdamW instead of Adam
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)  # Reduce LR every 10 epochs

# === Train the Model (50 Epochs) ===
num_epochs = 50
batch_size = 1024

for epoch in range(num_epochs):
    optimizer.zero_grad()
    outputs = model(X_train)
    loss = criterion(outputs, y_train)
    loss.backward()
    optimizer.step()
    scheduler.step()  # Adjust learning rate
    
    if (epoch + 1) % 5 == 0:  # Print loss every 5 epochs
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}")

# === Evaluate Model ===
with torch.no_grad():
    test_outputs = model(X_test)
    predicted = torch.argmax(test_outputs, axis=1)
    accuracy = (predicted == y_test).float().mean()
    print(f"Test Accuracy: {accuracy:.4f}")

Epoch [5/50], Loss: 1.3384
Epoch [10/50], Loss: 1.1675
Epoch [15/50], Loss: 1.0960
Epoch [20/50], Loss: 1.0363
Epoch [25/50], Loss: 1.0014
Epoch [30/50], Loss: 0.9705
Epoch [35/50], Loss: 0.9562
Epoch [40/50], Loss: 0.9436
Epoch [45/50], Loss: 0.9252
Epoch [50/50], Loss: 0.9262
Test Accuracy: 0.6622
